# 🧩 Esercizi Guidati — Agenti AI con Tool Calling

---

Questo notebook contiene **3 esercizi progressivi**.
Ogni esercizio fornisce:
- 🎯 L'**obiettivo** da raggiungere
- 📌 I **suggerimenti** passo per passo
- ✏️ Le **celle di codice** dove scrivere la soluzione

| Esercizio | Titolo | Difficoltà |
|---|---|---|
| **1** | Il tuo primo Tool personalizzato | 🟢 Facile |
| **2** | Agente con tool di ricerca e calcolo | 🟡 Medio |
| **3** | Mini-agente conversazionale multi-turno | 🔴 Avanzato |

> **Requisiti:** Ollama in esecuzione su `localhost:11434` con modello `llama3.2` o `qwen2.5`.

---
## ⚙️ Setup — Esegui questa cella prima di iniziare

In [1]:
%pip install -q langchain langchain-ollama langchain-core

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import os
import json
from typing import Any, List, Dict, Optional
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage, SystemMessage, BaseMessage
from langchain_core.tools import tool

MODEL_NAME = "llama3.2"  # oppure 'qwen2.5'
BASE_URL = "http://localhost:11434"

llm = ChatOllama(
    model=MODEL_NAME,
    base_url=BASE_URL,
    temperature=0, # Imposta la temperatura a 0 per risposte più deterministiche, serve a ridurre la casualità nelle risposte del modello
)

try:
    test_response = llm.invoke("Rispondi solo con: 'Pronto!'")
    print(f"✅ Connessione riuscita: {test_response.content.strip()}")
except Exception as e:
    print(f"❌ Errore di connessione: {e}")

c:\Users\A1618apulia\OneDrive - ITS Apulia Digital Maker\Desktop\GENERATIVEAI\.venv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.7.0) or chardet (6.0.0.post1)/charset_normalizer (3.5.1) doesn't match a supported version!
  warnings.warn(


✅ Connessione riuscita: 'Pronto!'


In [4]:
# Funzione di utilità per visualizzare la conversazione (identica al notebook 03)
def mostra_conversazione(messaggi: List[BaseMessage]):
    print("=" * 70)
    print("📜 CRONOLOGIA COMPLETA DELLA CONVERSAZIONE")
    print("=" * 70)
    for i, msg in enumerate(messaggi, 1):
        if isinstance(msg, SystemMessage):
            print(f"\n[{i}] ⚙️ SYSTEM:\n    {msg.content}")
        elif isinstance(msg, HumanMessage):
            print(f"\n[{i}] 👤 UTENTE:\n    {msg.content}")
        elif isinstance(msg, AIMessage):
            print(f"\n[{i}] 🤖 ASSISTENTE (AI):")
            if msg.content:
                print(f"    {msg.content}")
            if getattr(msg, 'tool_calls', None):
                for tc in msg.tool_calls:
                    print(f"    🛠️ [Tool Call] Funzione: '{tc['name']}' | Parametri: {json.dumps(tc['args'], ensure_ascii=False)}")
        elif isinstance(msg, ToolMessage):
            print(f"\n[{i}] 📥 RISULTATO TOOL (ID: {msg.tool_call_id}):\n    {msg.content}")
        else:
            print(f"\n[{i}] ❓ ALTRO ({type(msg).__name__}):\n    {msg.content}")
    print("\n" + "=" * 70)

---
## 🟢 Esercizio 1 — Il tuo primo Tool personalizzato

### 🎯 Obiettivo
Creare un tool `conta_parole` che:
1. Riceve un testo come stringa.
2. Conta il numero di parole nel testo.
3. Restituisce un messaggio con il conteggio.

Poi collegarlo al modello con `.bind_tools(...)` e testarlo chiedendo all'LLM di contare le parole in una frase.

### 📌 Suggerimenti
- Usa il decoratore `@tool` su una funzione Python con type annotations.
- Per contare le parole puoi usare `len(testo.split())`.
- La docstring è **fondamentale**: l'LLM la legge per capire quando usare il tool!
- Dopo aver definito il tool, usa `llm.bind_tools([conta_parole])` per collegarlo.
- Invoca il modello con una `HumanMessage` che richieda il conteggio delle parole.
- Controlla se la risposta contiene `tool_calls` e, in tal caso, esegui la funzione manualmente.

In [ ]:
# 📝 ESERCIZIO 1 — Parte A: Definisci il tool `conta_parole`

# TODO: Definisci qui la funzione `conta_parole` con il decoratore @tool
# La funzione deve:
#   - Accettare un parametro 'testo' di tipo str
#   - Restituire una stringa con il conteggio delle parole
#   - Avere una docstring descrittiva

# Scrivi il tuo codice qui:
@tool
def conta_parole(testo:str) -> str:
    """Conteggia il numero di parole in un testo fornito e restituisce il conteggio"""
    # Rimuove eventuali spazi biuanchi e divide il testo in parole
    parole = testo.strip().split()
    # Conta il numero di parole
    numero_parole= len(parole)
    # Restituisce il conteggio come stringa
    return {"conteggio_parole": numero_parole}

    

In [6]:
# 📝 ESERCIZIO 1 — Parte B: Collega il tool al modello e testa

llm_con_conta = llm.bind_tools([conta_parole])
prompt = 'Usa il tool conta_parole per contare le parole in questa frase: "L\'intelligenza artificiale sta trasformando il mondo del lavoro". Riporta il conteggio.'
messaggi = [HumanMessage(content=prompt)]
risposta = llm_con_conta.invoke(messaggi)
print("Risposta del modello:", risposta.content)
print("Tool calls:", risposta.tool_calls)

Risposta del modello: 
Tool calls: [{'name': 'conta_parole', 'args': {'testo': "L'intelligenza artificiale sta trasformando il mondo del lavoro"}, 'id': 'e6d533ba-9522-4c0a-a573-866f0111a864', 'type': 'tool_call'}]


In [7]:
# 📝 ESERCIZIO 1 — Parte C: Esegui il tool e ottieni la risposta finale

messaggi.append(risposta)
for tool_call in risposta.tool_calls:
    if tool_call["name"] == conta_parole.name:
        risultato_tool = conta_parole.invoke(tool_call["args"])
    else:
        risultato_tool = f"Tool non disponibile: {tool_call['name']}"
    messaggi.append(
        ToolMessage(content=str(risultato_tool), tool_call_id=tool_call["id"])
    )

if risposta.tool_calls:
    risposta_finale_es1 = llm_con_conta.invoke(messaggi)
    messaggi.append(risposta_finale_es1)
    print("Risposta finale:", risposta_finale_es1.content)
else:
    risposta_finale_es1 = risposta
    print("Il modello non ha richiesto il tool:", risposta.content)

mostra_conversazione(messaggi)

Risposta finale: La risposta è: Il testo contiene 8 parole.
📜 CRONOLOGIA COMPLETA DELLA CONVERSAZIONE

[1] 👤 UTENTE:
    Usa il tool conta_parole per contare le parole in questa frase: "L'intelligenza artificiale sta trasformando il mondo del lavoro". Riporta il conteggio.

[2] 🤖 ASSISTENTE (AI):
    🛠️ [Tool Call] Funzione: 'conta_parole' | Parametri: {"testo": "L'intelligenza artificiale sta trasformando il mondo del lavoro"}

[3] 📥 RISULTATO TOOL (ID: e6d533ba-9522-4c0a-a573-866f0111a864):
    Il testo contiene 8 parole.

[4] 🤖 ASSISTENTE (AI):
    La risposta è: Il testo contiene 8 parole.



---
## 🟡 Esercizio 2 — Agente con Tool di Ricerca e Calcolo

### 🎯 Obiettivo
Creare un mini-agente che ha a disposizione **due tool**:
1. `cerca_prodotto(nome: str) -> str` — simula una ricerca in un catalogo e restituisce il prezzo di un prodotto.
2. `calcola_sconto(prezzo: float, percentuale_sconto: float) -> str` — calcola il prezzo finale dopo uno sconto.

L'agente deve essere in grado di rispondere a: _"Quanto costa un laptop dopo il 15% di sconto?"_ usando **entrambi i tool in sequenza**.

### 📌 Suggerimenti
- Il catalogo può essere un semplice dizionario Python all'interno del tool.
- Il tool `cerca_prodotto` dovrebbe gestire il caso in cui il prodotto non esiste.
- Usa la funzione `esegui_agente` (che devi implementare qui) basandoti sul modello del notebook 03.
- Il loop agentico deve continuare finché l'AI non genera una risposta senza `tool_calls`.

In [8]:
# 📝 ESERCIZIO 2 — Parte A: Definisci i due tool

CATALOGO = {
    "laptop": 1200.0,
    "smartphone": 699.0,
    "tablet": 450.0,
    "cuffie": 89.0,
    "monitor": 320.0,
}


@tool
def cerca_prodotto(nome: str) -> str:
    """Cerca un prodotto nel catalogo e restituisce il suo prezzo in euro."""
    nome_normalizzato = nome.strip().casefold()
    prezzo = CATALOGO.get(nome_normalizzato)
    if prezzo is None:
        prodotti_disponibili = ", ".join(sorted(CATALOGO))
        return f"Prodotto '{nome}' non trovato. Disponibili: {prodotti_disponibili}."
    return f"Il prezzo del {nome_normalizzato} è {prezzo:.2f} euro."


@tool
def calcola_sconto(prezzo: float, percentuale_sconto: float) -> str:
    """Calcola e restituisce il prezzo in euro dopo uno sconto percentuale."""
    if prezzo < 0:
        return "Errore: il prezzo non può essere negativo."
    if not 0 <= percentuale_sconto <= 100:
        return "Errore: lo sconto deve essere compreso tra 0 e 100 percento."
    prezzo_finale = prezzo * (1 - percentuale_sconto / 100)
    return f"Prezzo finale dopo lo sconto: {prezzo_finale:.2f} euro."

In [22]:
# 📝 ESERCIZIO 2 — Parte B: Implementa la funzione `esegui_agente`

def esegui_agente(
    richiesta_utente: str,
    lista_tool: list,
    cronologia: Optional[List[BaseMessage]] = None,
    max_iterazioni: int = 6,
    verbose: bool = True,
) -> tuple[str, List[BaseMessage]]:
    """
    Esegue un loop agentico completo (Reasoning Loop / ReAct pattern).

    Il loop si chiude quando l'AI fornisce una risposta senza tool_calls,
    oppure al raggiungimento del numero massimo di iterazioni.

    Returns:
        Una tupla (risposta_finale_testo, cronologia_aggiornata)
    """
    # Creiamo il mappa nome_tool → funzione per l'esecuzione
    map_tool = {t.name: t for t in lista_tool}

    # Colleghiamo i tool al modello
    llm_con_tool = llm.bind_tools(lista_tool)

    # Inizializzazione cronologia: se non ne abbiamo una, partiamo da zero
    if cronologia is None:
        system_prompt = (
            "Usa un solo tool per volta e attendi il suo risultato prima di scegliere il passo successivo. "
            "Scegli il tool in base alla richiesta e passa solo valori coerenti con il tipo e il significato "
            "dei parametri descritti. Usa l'output dei tool per proseguire, senza inventare valori. "
            "Non chiedere spiegazioni all'utente quando puoi agire con le informazioni disponibili; "
            "completa il task e fornisci una risposta chiara."
        )
        if {"cerca_prodotto", "calcola_sconto"}.issubset(map_tool):
            system_prompt += (
                " Per calcolare lo sconto di un prodotto, chiama prima cerca_prodotto con il suo nome; "
                "poi chiama calcola_sconto usando il prezzo numerico restituito e la percentuale richiesta."
            )
        messaggi: List[BaseMessage] = [SystemMessage(content=system_prompt)]
    else:
        # Copiamo la cronologia esistente per non mutare la lista originale
        messaggi = list(cronologia)

    # Aggiungiamo il nuovo messaggio dell'utente
    messaggi.append(HumanMessage(content=richiesta_utente))

    if verbose:
        print(f"🎯 NUOVA RICHIESTA: {richiesta_utente}\n" + "=" * 60)

    # ---- REASONING LOOP ----
    for step in range(1, max_iterazioni + 1):
        if verbose:
            print(f"\n📍 [Ciclo {step}] L'agente sta ragionando...")

        # Invocazione del modello con la cronologia corrente
        risposta = llm_con_tool.invoke(messaggi)
        messaggi.append(risposta)

        # Condizione di uscita: nessuna tool call → risposta finale
        if not risposta.tool_calls:
            if verbose:
                print("🏁 L'agente ha completato il task!")
            return risposta.content, messaggi

        # Esecuzione del primo tool richiesto (un tool per ciclo)
        tc = risposta.tool_calls[0]
        t_name = tc["name"]
        t_args = tc["args"]
        t_id = tc.get("id", f"call_{step}")

        if verbose:
            print(f"  ⚡ Tool '{t_name}' → argomenti: {t_args}")

        # Lookup nella mappa e invocazione
        if t_name in map_tool:
            risultato = map_tool[t_name].invoke(t_args)
        else:
            risultato = f"Errore: Tool '{t_name}' non trovato tra quelli disponibili."

        if verbose:
            print(f"  👁️ Osservazione: {risultato}")

        # Aggiungiamo il risultato come ToolMessage per il ciclo successivo
        messaggi.append(ToolMessage(content=str(risultato), tool_call_id=t_id))

    # Limite iterazioni raggiunto
    return "⚠️ Limite massimo di iterazioni raggiunto senza risposta finale.", messaggi


print("✅ Funzione 'esegui_agente' definita.")

✅ Funzione 'esegui_agente' definita.


In [23]:
# 📝 ESERCIZIO 2 — Parte C: Testa l'agente

MODEL_NAME = "qwen2.5"
llm = ChatOllama(
    model=MODEL_NAME,
    base_url=BASE_URL,
    temperature=0,
)
lista_tool_es2 = [cerca_prodotto, calcola_sconto]

risposta_laptop, cronologia_laptop = esegui_agente(
    "Quanto costa un laptop dopo il 15% di sconto?",
    lista_tool_es2,
)
print("\nRisposta laptop:", risposta_laptop)

risposta_smartphone, cronologia_smartphone = esegui_agente(
    "Qual è il prezzo di uno smartphone con il 20% di sconto?",
    lista_tool_es2,
)
print("\nRisposta smartphone:", risposta_smartphone)

🎯 NUOVA RICHIESTA: Quanto costa un laptop dopo il 15% di sconto?

📍 [Ciclo 1] L'agente sta ragionando...
  ⚡ Tool 'cerca_prodotto' → argomenti: {'nome': 'laptop'}
  👁️ Osservazione: Il prezzo del laptop è 1200.00 euro.

📍 [Ciclo 2] L'agente sta ragionando...
  ⚡ Tool 'calcola_sconto' → argomenti: {'prezzo': 1200, 'percentuale_sconto': 15}
  👁️ Osservazione: Prezzo finale dopo lo sconto: 1020.00 euro.

📍 [Ciclo 3] L'agente sta ragionando...
🏁 L'agente ha completato il task!

Risposta laptop: Il laptop, dopo un sconto del 15%, costa 1020.00 euro.
🎯 NUOVA RICHIESTA: Qual è il prezzo di uno smartphone con il 20% di sconto?

📍 [Ciclo 1] L'agente sta ragionando...
  ⚡ Tool 'cerca_prodotto' → argomenti: {'nome': 'smartphone'}
  👁️ Osservazione: Il prezzo del smartphone è 699.00 euro.

📍 [Ciclo 2] L'agente sta ragionando...
  ⚡ Tool 'calcola_sconto' → argomenti: {'prezzo': 699, 'percentuale_sconto': 20}
  👁️ Osservazione: Prezzo finale dopo lo sconto: 559.20 euro.

📍 [Ciclo 3] L'agente sta rag

---
## 🔴 Esercizio 3 — Mini-Agente Conversazionale Multi-Turno

### 🎯 Obiettivo
Costruire un agente che:
1. Ha a disposizione **tre tool**: `scrivi_nota`, `leggi_nota`, `lista_note`.
2. Mantiene la **memoria della conversazione** tra i turni (passando la cronologia tra le chiamate).
3. Completa una sequenza di **3 operazioni in conversazioni separate**:
   - **Turno 1**: Scrivere una nota su "Python" con un breve contenuto.
   - **Turno 2**: Leggere la nota appena creata (l'agente deve ricordare il contesto del turno 1).
   - **Turno 3**: Chiedere all'agente la lista di tutte le note esistenti.

### 📌 Suggerimenti
- Le note possono essere salvate come file `.txt` in una cartella `note/`.
- `scrivi_nota(titolo: str, contenuto: str)` → crea `note/{titolo}.txt`.
- `leggi_nota(titolo: str)` → legge `note/{titolo}.txt`.
- `lista_note()` → elenca tutti i file in `note/` (questo tool non ha argomenti obbligatori).
- La **chiave** dell'esercizio è passare `cronologia` al secondo e terzo turno per mantenere la memoria!
- Usa `mostra_conversazione()` alla fine per visualizzare l'intera storia dei 3 turni.

In [16]:
# 📝 ESERCIZIO 3 — Parte A: Definisci i tre tool per la gestione note

os.makedirs("note", exist_ok=True)


def _percorso_nota(titolo: str) -> Optional[str]:
    """Normalizza il titolo e limita i file alla cartella note/."""
    nome_file = os.path.basename(titolo.replace("\\", "/")).strip()
    if nome_file.lower().endswith(".txt"):
        nome_file = nome_file[:-4]
    if nome_file in {"", ".", ".."}:
        return None
    return os.path.join("note", f"{nome_file}.txt")


@tool
def scrivi_nota(titolo: str, contenuto: str) -> str:
    """Crea o aggiorna una nota con il titolo e il contenuto indicati."""
    percorso = _percorso_nota(titolo)
    if percorso is None:
        return "Errore: il titolo della nota non è valido."
    with open(percorso, "w", encoding="utf-8") as file:
        file.write(contenuto)
    return f"Nota '{os.path.basename(percorso)}' salvata correttamente."


@tool
def leggi_nota(titolo: str) -> str:
    """Legge e restituisce il contenuto della nota con il titolo indicato."""
    percorso = _percorso_nota(titolo)
    if percorso is None:
        return "Errore: il titolo della nota non è valido."
    if not os.path.exists(percorso):
        return f"La nota '{titolo}' non esiste."
    with open(percorso, "r", encoding="utf-8") as file:
        return file.read()


@tool
def lista_note() -> str:
    """Elenca tutti i file .txt presenti nella cartella delle note."""
    nomi_note = sorted(
        nome for nome in os.listdir("note") if nome.lower().endswith(".txt")
    )
    if not nomi_note:
        return "Non ci sono note salvate."
    return "Note disponibili: " + ", ".join(nomi_note)

In [24]:
# 📝 ESERCIZIO 3 — Parte B: Turno 1 — Scrivi una nota

lista_tool_es3 = [scrivi_nota, leggi_nota, lista_note]

richiesta_turno_1 = (
    "Scrivi una nota dal titolo 'Python' con questo contenuto: "
    "'Python è un linguaggio di programmazione versatile, molto usato in AI e data science.'"
)

risposta_t1, cronologia_completa = esegui_agente(
    richiesta_turno_1,
    lista_tool_es3,
)
print("\n💬 Risposta Turno 1:")
print(risposta_t1)


🎯 NUOVA RICHIESTA: Scrivi una nota dal titolo 'Python' con questo contenuto: 'Python è un linguaggio di programmazione versatile, molto usato in AI e data science.'

📍 [Ciclo 1] L'agente sta ragionando...
  ⚡ Tool 'scrivi_nota' → argomenti: {'titolo': 'Python', 'contenuto': 'Python è un linguaggio di programmazione versatile, molto usato in AI e data science.'}
  👁️ Osservazione: Nota 'Python.txt' salvata correttamente.

📍 [Ciclo 2] L'agente sta ragionando...
🏁 L'agente ha completato il task!

💬 Risposta Turno 1:
La nota dal titolo "Python" è stata salvata correttamente.


In [25]:
# 📝 ESERCIZIO 3 — Parte C: Turno 2 — Leggi la nota (con memoria del Turno 1)

richiesta_turno_2 = "Puoi leggere la nota che hai appena scritto su Python?"
risposta_t2, cronologia_completa = esegui_agente(
    richiesta_turno_2,
    lista_tool_es3,
    cronologia=cronologia_completa,
)
print("\n💬 Risposta Turno 2:")
print(risposta_t2)

🎯 NUOVA RICHIESTA: Puoi leggere la nota che hai appena scritto su Python?

📍 [Ciclo 1] L'agente sta ragionando...
  ⚡ Tool 'leggi_nota' → argomenti: {'titolo': 'Python'}
  👁️ Osservazione: Python è un linguaggio di programmazione versatile, molto usato in AI e data science.

📍 [Ciclo 2] L'agente sta ragionando...
🏁 L'agente ha completato il task!

💬 Risposta Turno 2:
Ecco il contenuto della nota "Python": "Python è un linguaggio di programmazione versatile, molto usato in AI e data science."


In [26]:
# 📝 ESERCIZIO 3 — Parte D: Turno 3 — Lista di tutte le note

richiesta_turno_3 = "Quante note hai salvato finora? Elencale."
risposta_t3, cronologia_completa = esegui_agente(
    richiesta_turno_3,
    lista_tool_es3,
    cronologia=cronologia_completa,
)
print("\n💬 Risposta Turno 3:")
print(risposta_t3)

mostra_conversazione(cronologia_completa)

🎯 NUOVA RICHIESTA: Quante note hai salvato finora? Elencale.

📍 [Ciclo 1] L'agente sta ragionando...
  ⚡ Tool 'lista_note' → argomenti: {}
  👁️ Osservazione: Note disponibili: Python.txt

📍 [Ciclo 2] L'agente sta ragionando...
🏁 L'agente ha completato il task!

💬 Risposta Turno 3:
Finora ho salvato 1 nota. Ecco l'elenco delle note disponibili: Python.txt.
📜 CRONOLOGIA COMPLETA DELLA CONVERSAZIONE

[1] ⚙️ SYSTEM:
    Usa un solo tool per volta e attendi il suo risultato prima di scegliere il passo successivo. Scegli il tool in base alla richiesta e passa solo valori coerenti con il tipo e il significato dei parametri descritti. Usa l'output dei tool per proseguire, senza inventare valori. Non chiedere spiegazioni all'utente quando puoi agire con le informazioni disponibili; completa il task e fornisci una risposta chiara.

[2] 👤 UTENTE:
    Scrivi una nota dal titolo 'Python' con questo contenuto: 'Python è un linguaggio di programmazione versatile, molto usato in AI e data science.'

[

In [28]:
# Visualizzazione dell'intera cronologia dei 3 turni
print(f"Totale messaggi in cronologia: {len(cronologia_completa)}\n")
mostra_conversazione(messaggi=cronologia_completa)

Totale messaggi in cronologia: 13

📜 CRONOLOGIA COMPLETA DELLA CONVERSAZIONE

[1] ⚙️ SYSTEM:
    Usa un solo tool per volta e attendi il suo risultato prima di scegliere il passo successivo. Scegli il tool in base alla richiesta e passa solo valori coerenti con il tipo e il significato dei parametri descritti. Usa l'output dei tool per proseguire, senza inventare valori. Non chiedere spiegazioni all'utente quando puoi agire con le informazioni disponibili; completa il task e fornisci una risposta chiara.

[2] 👤 UTENTE:
    Scrivi una nota dal titolo 'Python' con questo contenuto: 'Python è un linguaggio di programmazione versatile, molto usato in AI e data science.'

[3] 🤖 ASSISTENTE (AI):
    🛠️ [Tool Call] Funzione: 'scrivi_nota' | Parametri: {"titolo": "Python", "contenuto": "Python è un linguaggio di programmazione versatile, molto usato in AI e data science."}

[4] 📥 RISULTATO TOOL (ID: a637cbed-5698-42f3-adaf-5f6069227e16):
    Nota 'Python.txt' salvata correttamente.

[5] 🤖 ASSI

In [27]:
# Verifica finale: il file esiste davvero sul disco?
percorso_nota = "note/Python.txt"
if os.path.exists(percorso_nota):
    print(f"✅ Il file '{percorso_nota}' esiste!")
    with open(percorso_nota, "r", encoding="utf-8") as f:
        print("--- Contenuto ---")
        print(f.read())
        print("-----------------")
else:
    print(f"⚠️ Il file '{percorso_nota}' non è stato trovato.")

✅ Il file 'note/Python.txt' esiste!
--- Contenuto ---
Python è un linguaggio di programmazione versatile, molto usato in AI e data science.
-----------------


---
## 🏁 Ricapitolo degli Esercizi

| Esercizio | Cosa hai imparato |
|---|---|
| **1 — Tool base** | Creare un tool con `@tool`, type annotations, docstring; binding e invocazione manuale del tool |
| **2 — Multi-tool sequenziale** | Reasoning loop multi-step, map dei tool, tool che si concatenano |
| **3 — Multi-turno con memoria** | Persistenza della `cronologia` tra turni, tool senza argomenti, agente con stato |

> 💡 **Prossimi passi**: consulta il notebook `04_soluzioni_agenti_ai.ipynb` per confrontare la tua soluzione con quella di riferimento.